# 02 · Seleção de atributos por votação

**Objetivo.** Escolher, no ciclo corrente, um único conjunto de atributos usado pelos dois modelos competidores. Três métodos independentes votam e a regra de votação elege o conjunto.

**Entradas.**
- `reports/ciclo_atual.json`: `cycle_id` e `data_version` abertos pelo notebook 01.
- `data/processed/dataset_processado.parquet`: só a partição **treino** é usada.

**Saídas.**
- `data/processed/features_eleitas.json`: atributos eleitos, lido pelos notebooks 03 e 04.
- `reports/evidencias/<cycle_id>/02_votacao_features.csv`: uma linha por (método, atributo) com `cycle_id`, `data_version`, `method`, `feature`, `score`, `rank`, `vote` e `final_selection`.
- MLflow: run filho `votacao_features` com a tabela de votos.

**Metodologia.** Os atributos candidatos (29: os 23 originais e os 6 derivados) são imputados pela mediana e padronizados com parâmetros do treino. Cada método escolhe os seus melhores atributos:

| Método | Critério | Quantos escolhe |
|---|---|---|
| 1. `SelectKBest(f_classif)` | Estatística F da ANOVA entre o atributo e a classe. Mede a separação linear das médias | `features.k_best` |
| 2. Importância de `RandomForestClassifier` | Redução média de impureza de Gini. Capta não linearidades e interações | `features.random_forest_features` |
| 3. `RFE` com `LogisticRegression` | Eliminação recursiva do atributo com menor coeficiente absoluto. Considera o efeito conjunto | `features.rfe_features` |

**Regra de votação.** Um atributo é eleito quando pelo menos `features.voting.minimum_votes` métodos o selecionam (padrão: 2 de 3).

**Decisões.**
- A votação ocorre sobre as colunas originais, e não sobre as colunas após o one-hot. Assim o conjunto eleito é legível e o mesmo nos dois modelos. Nesta triagem, `SEX`, `EDUCATION` e `MARRIAGE` entram como códigos inteiros. Os modelos aplicam one-hot depois.
- Validação e teste não participam. Usá-los aqui vazaria informação para a competição.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

import mlflow

from src import features as fs, preprocessamento as pp
from src.utils import cycle_run, load_config, load_cycle

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

In [2]:
ctx = load_cycle(cfg, ROOT)
dataset = pp.load_processed(cfg, ctx, ROOT)
candidates = pp.feature_columns(cfg, dataset)
X_train, y_train = pp.split_xy(dataset, "treino", candidates, cfg)

print("cycle_id     :", ctx.cycle_id)
print("data_version :", ctx.data_version)
print("treino       :", X_train.shape, "| candidatos:", len(candidates))

cycle_id     : ciclo_20261116T223819Z_2db70707
data_version : dv_2db70707140b1359
treino       : (14475, 29) | candidatos: 29


## 1. Execução dos três métodos e votação

In [3]:
result = fs.run_feature_voting(X_train, y_train, cfg, ctx)

by_method = result.records.pivot(index="feature", columns="method", values=["rank", "vote"])
by_method.columns = [f"{m}__{v}" for v, m in by_method.columns]
overview = (by_method.join(result.tally.set_index("feature"))
            .sort_values(["votes", "select_k_best_f_classif__rank"], ascending=[False, True]))
display(overview)

,random_forest_importance__rank,rfe_logistic_regression__rank,select_k_best_f_classif__rank,random_forest_importance__vote,rfe_logistic_regression__vote,select_k_best_f_classif__vote,votes,elected
feature,,,,,,,,
MESES_EM_ATRASO,2,1,1,True,True,True,3,True
ATRASO_MAXIMO,3,1,2,True,True,True,3,True
PAY_0,1,1,3,True,True,True,3,True
LIMIT_BAL,9,1,9,True,True,True,3,True
PAY_2,4,4,4,True,False,True,2,True
PAY_3,16,1,5,False,True,True,2,True
PAY_6,26,1,8,False,True,True,2,True
UTILIZACAO_MEDIA,7,6,10,True,False,True,2,True
PAY_AMT1,6,1,12,True,True,False,2,True


## 2. Pontuação de cada método

`score` depende do método: estatística F (método 1), importância de Gini (método 2) e coeficiente absoluto final (método 3, que vale 0 para os eliminados). `rank` = 1 é o melhor atributo de cada método.

In [4]:
for method, part in result.records.groupby("method"):
    print(method)
    display(part.sort_values("rank")[["feature", "score", "rank", "vote"]].head(12).reset_index(drop=True))

random_forest_importance


,feature,score,rank,vote
0,PAY_0,0.149886,1,True
1,MESES_EM_ATRASO,0.115574,2,True
2,ATRASO_MAXIMO,0.113542,3,True
3,PAY_2,0.058493,4,True
4,UTILIZACAO_ATUAL,0.038068,5,True
5,PAY_AMT1,0.033317,6,True
6,UTILIZACAO_MEDIA,0.032698,7,True
7,PAY_AMT2,0.030416,8,True
8,LIMIT_BAL,0.029108,9,True
9,TENDENCIA_FATURA,0.028742,10,True


rfe_logistic_regression


,feature,score,rank,vote
0,LIMIT_BAL,0.188168,1,True
1,PAY_0,0.201532,1,True
2,PAY_3,0.238004,1,True
3,BILL_AMT1,0.240380,1,True
4,BILL_AMT2,0.474519,1,True
5,BILL_AMT5,0.180230,1,True
6,PAY_6,0.193098,1,True
7,PAY_AMT1,0.306035,1,True
8,ATRASO_MAXIMO,0.226230,1,True
9,MESES_EM_ATRASO,0.769534,1,True


select_k_best_f_classif


,feature,score,rank,vote
0,MESES_EM_ATRASO,2720.228690,1,True
1,ATRASO_MAXIMO,1730.054836,2,True
2,PAY_0,1728.230836,3,True
3,PAY_2,1105.055419,4,True
4,PAY_3,838.433833,5,True
5,PAY_4,723.555089,6,True
6,PAY_5,677.947939,7,True
7,PAY_6,523.423684,8,True
8,LIMIT_BAL,344.839920,9,True
9,UTILIZACAO_MEDIA,165.389000,10,True


## 3. Conjunto eleito

In [5]:
payload = fs.save_voting(result, ctx, cfg, ROOT)
print(f"{len(result.elected)} atributos eleitos (mínimo de {payload['minimum_votes']} votos):")
for feature in result.elected:
    print(f"  {feature:<18} {payload['votes'][feature]} votos")

agreement = result.tally["votes"].value_counts().sort_index(ascending=False)
display(agreement.rename_axis("votos").rename("n_atributos").to_frame())

9 atributos eleitos (mínimo de 2 votos):
  LIMIT_BAL          3 votos
  PAY_0              3 votos
  PAY_2              2 votos
  PAY_3              2 votos
  PAY_6              2 votos
  PAY_AMT1           2 votos
  UTILIZACAO_MEDIA   2 votos
  MESES_EM_ATRASO    3 votos
  ATRASO_MAXIMO      3 votos


,n_atributos
votos,
3,4
2,5
1,8
0,12


## 4. Registro no MLflow

In [6]:
with cycle_run(ctx, cfg, stage="votacao_features", root=ROOT):
    mlflow.log_params({"methods": ",".join(fs.SELECTORS), "minimum_votes": payload["minimum_votes"],
                       "k_best": cfg["features"]["k_best"], "rfe_features": cfg["features"]["rfe_features"],
                       "random_forest_features": cfg["features"]["random_forest_features"],
                       "n_candidates": len(candidates), "n_train": len(X_train)})
    mlflow.log_metric("n_elected", len(result.elected))
    mlflow.log_dict({"elected_features": result.elected, "votes": payload["votes"]}, "features_eleitas.json")
    mlflow.log_table(result.records, "votacao_features.json")
print("run votacao_features registrado no ciclo", ctx.cycle_id)

run votacao_features registrado no ciclo ciclo_20261116T223819Z_2db70707


## Conclusões

- Os três métodos rodaram no mesmo ciclo, sobre os mesmos dados de treino, e participaram de uma única votação.
- Cada voto ficou registrado com `cycle_id` e `data_version`, no CSV de evidência e no MLflow.
- Os atributos de histórico de pagamento (`PAY_*` e os derivados de atraso) costumam concentrar os votos. Isso é coerente com a literatura de risco de crédito: o comportamento recente de pagamento é o melhor preditor de inadimplência.
- Um lote novo muda o treino e, portanto, pode mudar o conjunto eleito. Por isso a votação roda de novo em todo ciclo.

**Próximo passo:** `03_treino_competicao.ipynb` treina os modelos A e B com exatamente estes atributos.